In [1]:
"""
FDA NDC Directory Loader (Product + Package)
--------------------------------------------
Downloads the daily NDC text files from FDA,
builds standardized labeler_id and 11-digit NDC,
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

import pandas as pd
import requests
import zipfile
import io
from pathlib import Path
from datetime import datetime
from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

# Download text zip
url = "https://www.accessdata.fda.gov/cder/ndctext.zip"
print("Downloading NDC text zip...")
response = requests.get(url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files in zip:")
    for name in z.namelist():
        print(" ", name)

    # --- PACKAGE ---
    package_file = [n for n in z.namelist() if "package" in n.lower() and n.endswith(".txt")][0]
    with z.open(package_file) as f:
        df_package = pd.read_csv(f, sep="\t", dtype=str, low_memory=False, encoding="latin-1")

    # --- PRODUCT ---
    product_file = [n for n in z.namelist() if "product" in n.lower() and n.endswith(".txt")][0]
    with z.open(product_file) as f:
        df_product = pd.read_csv(f, sep="\t", dtype=str, low_memory=False, encoding="latin-1")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc):
    parts = ndc.astype(str).str.split("-", expand=True)
    if parts.shape[1] == 3:
        labeler  = parts[0].str.zfill(5)
        product  = parts[1].str.zfill(4)
        package  = parts[2].str.zfill(2)
        return labeler + product + package
    return pd.Series([None] * len(ndc))

# ---------- Add columns ----------
load_ts = datetime.now()

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"]      = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"]  = load_ts

df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"]  = load_ts

# ---------- Save ----------

# Comment Out xlsx files only use for viewability
df_package.to_excel(out_path / "ndc_package.xlsx", index=False)
df_package.to_parquet(out_path / "ndc_package.parquet", index=False)

# Comment Out xlsx files only use for viewability
df_product.to_excel(out_path / "ndc_product.xlsx", index=False)
df_product.to_parquet(out_path / "ndc_product.parquet", index=False)

print(f"Package: {df_package.shape[0]:,} rows → ndc_package.xlsx / .parquet")
print(f"Product: {df_product.shape[0]:,} rows → ndc_product.xlsx / .parquet")
print("Done.")

Files in zip:
  package.txt
  product.txt
Package: 219,121 rows → ndc_package.xlsx / .parquet
Product: 116,594 rows → ndc_product.xlsx / .parquet
Done.


In [2]:
"""
FDA Compounded Drugs NDC Directory Loader
-----------------------------------------
Downloads the daily compounders NDC file from FDA,
builds standardized labeler_id and 11-digit NDC,
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/compounders_ndc_directory.zip"
zip_path = out_path / "compounders_ndc_directory.zip"

print("Downloading compound NDC file...")
resp = requests.get(url, timeout=120)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("compounders_ndc_directory.txt", out_path)

txt_path = out_path / "compounders_ndc_directory.txt"

# ---------- Load ----------
df = pd.read_csv(txt_path, sep="\t", dtype=str, low_memory=False)
print(f"Loaded {len(df):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

df["labeler_id"] = build_labeler(df["PRODUCTNDC"])
df["ndc11"] = build_ndc11(df["NDCPACKAGECODE"])
df["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
xlsx_path = out_path / "ndc_compound.xlsx"
parquet_path = out_path / "ndc_compound.parquet"

df.to_excel(xlsx_path, index=False)
df.to_parquet(parquet_path, index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(xlsx_path, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"Compound: {df.shape[0]:,} rows → ndc_compound.xlsx / .parquet")
print("Done.")

Loaded 4,836 rows
Compound: 4,836 rows → ndc_compound.xlsx / .parquet
Done.


In [3]:
"""
FDA Unfinished Drugs NDC Directory Loader
-----------------------------------------
Downloads the daily unfinished NDC files from FDA,
builds standardized labeler_id (both files) and 11-digit NDC (package only),
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {out_path.resolve()}")

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/ndc_unfinished.zip"
zip_path = out_path / "ndc_unfinished.zip"

print("Downloading unfinished NDC files...")
resp = requests.get(url, timeout=120)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("unfinished_product.txt", out_path)
    z.extract("unfinished_package.txt", out_path)
    product_modified = datetime(*z.getinfo("unfinished_product.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")
    package_modified = datetime(*z.getinfo("unfinished_package.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")

# ---------- Load (encoding-safe) ----------
def safe_read(path):
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, sep="\t", dtype=str, low_memory=False, encoding=enc)
        except Exception as e:
            print(f"  {enc} failed: {e}")
    raise RuntimeError(f"Could not read {path}")

print("Loading product...")
df_product = safe_read(out_path / "unfinished_product.txt")
print("Loading package...")
df_package = safe_read(out_path / "unfinished_package.txt")

print(f"Product: {len(df_product):,} rows")
print(f"Package: {len(df_package):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

# ---------- Enrich ----------
df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"] = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
product_xlsx = out_path / "ndc_unfinished_product.xlsx"
package_xlsx = out_path / "ndc_unfinished_package.xlsx"

df_product.to_excel(product_xlsx, index=False)
df_product.to_parquet(out_path / "ndc_unfinished_product.parquet", index=False)

df_package.to_excel(package_xlsx, index=False)
df_package.to_parquet(out_path / "ndc_unfinished_package.parquet", index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(product_xlsx, ["PRODUCTNDC", "labeler_id"])
force_text_columns(package_xlsx, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"\nFiles written to: {out_path.resolve()}")
print(f"  {product_xlsx.name}")
print(f"  ndc_unfinished_product.parquet")
print(f"  {package_xlsx.name}")
print(f"  ndc_unfinished_package.parquet")
print("Done.")

Output folder: C:\Users\Patri\OneDrive\Desktop\NDC_DataMart
Loading product...
  utf-8 failed: 'utf-8' codec can't decode byte 0xf3 in position 46: invalid continuation byte
Loading package...
Product: 21,668 rows
Package: 37,156 rows

Files written to: C:\Users\Patri\OneDrive\Desktop\NDC_DataMart
  ndc_unfinished_product.xlsx
  ndc_unfinished_product.parquet
  ndc_unfinished_package.xlsx
  ndc_unfinished_package.parquet
Done.


In [4]:
"""
FDA Excluded Drugs NDC Directory Loader
---------------------------------------
Downloads the daily excluded NDC files from FDA,
builds standardized labeler_id (both files) and 11-digit NDC (package only),
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {out_path.resolve()}")

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/ndc_excluded.zip"
zip_path = out_path / "ndc_excluded.zip"

print("Downloading excluded NDC files...")
resp = requests.get(url, timeout=180)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("Products_excluded.txt", out_path)
    z.extract("Packages_excluded.txt", out_path)
    product_modified = datetime(*z.getinfo("Products_excluded.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")
    package_modified = datetime(*z.getinfo("Packages_excluded.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")

# ---------- Load (encoding-safe) ----------
def safe_read(path):
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, sep="\t", dtype=str, low_memory=False, encoding=enc)
        except Exception as e:
            print(f"  {enc} failed: {e}")
    raise RuntimeError(f"Could not read {path}")

print("Loading product...")
df_product = safe_read(out_path / "Products_excluded.txt")
print("Loading package...")
df_package = safe_read(out_path / "Packages_excluded.txt")

print(f"Product: {len(df_product):,} rows")
print(f"Package: {len(df_package):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

# ---------- Enrich ----------
df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"] = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
product_xlsx = out_path / "ndc_excluded_product.xlsx"
package_xlsx = out_path / "ndc_excluded_package.xlsx"

df_product.to_excel(product_xlsx, index=False)
df_product.to_parquet(out_path / "ndc_excluded_product.parquet", index=False)

df_package.to_excel(package_xlsx, index=False)
df_package.to_parquet(out_path / "ndc_excluded_package.parquet", index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(product_xlsx, ["PRODUCTNDC", "labeler_id"])
force_text_columns(package_xlsx, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"\nFiles written to: {out_path.resolve()}")
print(f"  {product_xlsx.name}")
print(f"  ndc_excluded_product.parquet")
print(f"  {package_xlsx.name}")
print(f"  ndc_excluded_package.parquet")
print("Done.")

Output folder: C:\Users\Patri\OneDrive\Desktop\NDC_DataMart
Loading product...
  utf-8 failed: 'utf-8' codec can't decode byte 0x96 in position 7: invalid start byte
Loading package...
Product: 207,559 rows
Package: 386,358 rows

Files written to: C:\Users\Patri\OneDrive\Desktop\NDC_DataMart
  ndc_excluded_product.xlsx
  ndc_excluded_product.parquet
  ndc_excluded_package.xlsx
  ndc_excluded_package.parquet
Done.


In [5]:
import duckdb
from pathlib import Path
from config import out_folder

con = duckdb.connect()
out_path = Path(out_folder)

con.execute(f"""
CREATE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');
""")

query = """
SELECT 
    p.ndc11,
    p.NDCPACKAGECODE,
    COALESCE(p.PACKAGEDESCRIPTION, '')          AS package_description,
    p.STARTMARKETINGDATE                        AS package_start_date,
    COALESCE(p.ENDMARKETINGDATE, '')            AS package_end_date,
    p.labeler_id,
    p.load_date,
    
    pr.PRODUCTNDC,
    COALESCE(pr.PROPRIETARYNAME, '')            AS proprietary_name,
    COALESCE(pr.NONPROPRIETARYNAME, '')         AS nonproprietary_name,
    COALESCE(pr.DOSAGEFORMNAME, '')             AS dosage_form,
    COALESCE(pr.ROUTENAME, '')                  AS route,
    COALESCE(pr.SUBSTANCENAME, '')              AS substance,
    COALESCE(pr.ACTIVE_NUMERATOR_STRENGTH, '')  AS strength,
    COALESCE(pr.ACTIVE_INGRED_UNIT, '')         AS strength_unit,
    COALESCE(pr.PHARM_CLASSES, '')              AS pharm_classes,
    COALESCE(pr.DEASCHEDULE, '')                AS dea_schedule,
    COALESCE(pr.LABELERNAME, '')                AS labeler_name,
    COALESCE(pr.MARKETINGCATEGORYNAME, '')      AS marketing_category,
    pr.STARTMARKETINGDATE                       AS product_start_date,
    COALESCE(pr.ENDMARKETINGDATE, '')           AS product_end_date

FROM package p
LEFT JOIN product pr 
    ON p.PRODUCTID = pr.PRODUCTID
"""

df = con.execute(query).df()
print(f"Joined rows: {len(df):,}")
df.head(1)

Joined rows: 219,121


,ndc11,NDCPACKAGECODE,package_description,package_start_date,package_end_date,labeler_id,load_date,PRODUCTNDC,proprietary_name,nonproprietary_name,...,route,substance,strength,strength_unit,pharm_classes,dea_schedule,labeler_name,marketing_category,product_start_date,product_end_date
0,65162025003,65162-250-03,30 TABLET in 1 BOTTLE (65162-250-03),20190701,,65162,2026-11-08 18:29:57.712951,65162-250,Chlorthalidone,Chlorthalidone,...,ORAL,CHLORTHALIDONE,50,mg/1,"Increased Diuresis [PE], Thiazide-like Diureti...",,Amneal Pharmaceuticals LLC,ANDA,20190701,
